In [ ]:
# --- Auto-install missing dependencies ---
import sys, subprocess

def install(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", pkg])

for package in ["opencv-python", "joblib", "scikit-learn", "scikit-image", "numpy", "pillow", "ipywidgets"]:
    try:
        __import__(package.split("-")[0])
    except ImportError:
        install(package)

print("✅ All required packages are installed and ready!")


In [2]:
# ==========================================
# SHIP DETECTION USING TRAINED RF CLASSIFIER
# ==========================================

import os
import joblib
import cv2
import numpy as np
from skimage.feature import graycomatrix, graycoprops
from IPython.display import display
import ipywidgets as widgets
from PIL import Image
import io

print("✅ Detection environment initialized!")

ModuleNotFoundError: No module named 'cv2'

In [ ]:
# === Load model and scaler safely ===

model_path = "models/random_forest_ship_classifier.pkl"
scaler_path = "models/minmax_scaler.pkl"

if not os.path.exists(model_path) or not os.path.exists(scaler_path):
    raise FileNotFoundError(
        f"❌ Model or scaler not found.\n"
        f"Expected:\n - {model_path}\n - {scaler_path}\n"
        f"Make sure you've trained and saved them in 'Traditional.ipynb'."
    )

final_rf = joblib.load(model_path)
scaler = joblib.load(scaler_path)
print("✅ Model and scaler loaded successfully!")
print(f"📁 Model:  {os.path.abspath(model_path)}")
print(f"📁 Scaler: {os.path.abspath(scaler_path)}")


In [ ]:
# === Feature extraction functions ===

def extract_hu_moments(img):
    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    moments = cv2.moments(gray)
    hu = cv2.HuMoments(moments).flatten()
    hu = -np.sign(hu) * np.log10(np.abs(hu) + 1e-7)
    return hu

def extract_haralick_features(img):
    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    glcm = graycomatrix(
        gray, [1], [0, np.pi/4, np.pi/2, 3*np.pi/4],
        symmetric=True, normed=True
    )
    props = []
    for prop in ['contrast', 'dissimilarity', 'homogeneity', 'ASM', 'energy', 'correlation']:
        props.append(graycoprops(glcm, prop).mean())
    return np.array(props)

def extract_color_histogram(img, bins=(8,8,8)):
    hsv = cv2.cvtColor(img, cv2.COLOR_RGB2HSV)
    hist = cv2.calcHist([hsv],[0,1,2],None,bins,[0,180,0,256,0,256])
    hist = cv2.normalize(hist, hist).flatten()
    return hist


In [ ]:
# === Robust prediction helper (auto-detect feature set) ===
import numpy as np
import cv2

def _compute_all_feature_groups(img_rgb, img_size=256):
    """
    Compute Hu, Haralick, and Color-Hist features (1D numpy arrays).
    img_rgb: HxWx3 RGB numpy array
    """
    img_rs = cv2.resize(img_rgb, (img_size, img_size))
    hu = extract_hu_moments(img_rs)            # length 7
    ha = extract_haralick_features(img_rs)     # length 6
    his = extract_color_histogram(img_rs)      # length depends on bins (512 if bins=(8,8,8))
    return hu, ha, his

def rf_predict_image_array_auto(img_rgb, model, scaler, img_size=256):
    """
    Auto-detect which feature set was used during training by comparing scaler.n_features_in_.
    Returns: (label, prob, used_feature_set)
    label: 0/1, prob: ship probability float, used_feature_set: str like "Ha+His"
    """
    if not hasattr(scaler, "n_features_in_"):
        raise RuntimeError("Scaler missing attribute 'n_features_in_'. Was it fit before saving?")

    expected = int(scaler.n_features_in_)
    hu, ha, his = _compute_all_feature_groups(img_rgb, img_size=img_size)

    # possible combos
    combos = {
        "Hu": hu.reshape(1, -1),
        "Ha": ha.reshape(1, -1),
        "His": his.reshape(1, -1),
        "Hu+Ha": np.hstack((hu, ha)).reshape(1, -1),
        "Ha+His": np.hstack((ha, his)).reshape(1, -1),
        "Hu+His": np.hstack((hu, his)).reshape(1, -1),
        "All": np.hstack((hu, ha, his)).reshape(1, -1)
    }

    matched_name = None
    matched_feats = None
    for name, feats in combos.items():
        if feats.shape[1] == expected:
            matched_name = name
            matched_feats = feats
            break

    if matched_name is None:
        # helpful diagnostic
        sizes = {k: v.shape[1] for k, v in combos.items()}
        raise ValueError(
            f"No feature combination matches scaler.n_features_in_ = {expected}.\n"
            f"Computed feature sizes: {sizes}\n"
            f"Ensure histogram bins and feature functions match training."
        )

    # scale and predict
    feats_scaled = scaler.transform(matched_feats)
    prob = float(model.predict_proba(feats_scaled)[0, 1])
    label = int(model.predict(feats_scaled)[0])

    return label, prob, matched_name

# convenience alias to keep previous code compatibility
rf_predict_image_array = lambda img_rgb, model, scaler, img_size=256: rf_predict_image_array_auto(img_rgb, model, scaler, img_size)[:2]


In [ ]:
# === Upload widget that uses the auto-detect helper ===
from IPython.display import display
import ipywidgets as widgets
from PIL import Image
import io

def _get_uploaded_image_bytes(uploader):
    val = uploader.value
    if val is None:
        return None, None
    if isinstance(val, dict):
        entry = next(iter(val.values()))
        return entry.get('metadata', {}).get('name', 'uploaded_image'), entry.get('content')
    if isinstance(val, (list, tuple)):
        first = val[0]
        if isinstance(first, dict) and 'content' in first:
            return first.get('metadata', {}).get('name', 'uploaded_image'), first.get('content')
        if isinstance(first, (list, tuple)) and len(first) >= 2:
            if isinstance(first[0], str) and isinstance(first[1], (bytes, bytearray)):
                return first[0], first[1]
            if isinstance(first[1], (bytes, bytearray)):
                return "uploaded_image", first[1]
        if isinstance(first, (bytes, bytearray)):
            return "uploaded_image", first
    return None, None

def upload_and_predict_widget(img_size=256):
    # require model and scaler present in notebook globals
    try:
        final_rf  # noqa
        scaler    # noqa
        rf_predict_image_array_auto  # noqa
    except NameError:
        print("Model/scaler/helper not found. Ensure you ran the 'Load Model' and 'Prediction Helper' cells.")
        return

    uploader = widgets.FileUpload(accept='image/*', multiple=False)
    display(uploader)
    out = widgets.Output()
    display(out)

    def on_upload_change(change):
        with out:
            out.clear_output()
            fname, img_bytes = _get_uploaded_image_bytes(uploader)
            if img_bytes is None:
                print("⚠️ No valid image uploaded.")
                return
            try:
                pil = Image.open(io.BytesIO(img_bytes)).convert('RGB')
            except Exception as exc:
                print("Failed to read image:", exc)
                return

            rgb = np.array(pil)
            try:
                label, prob, used = rf_predict_image_array_auto(rgb, final_rf, scaler, img_size=img_size)
            except Exception as exc:
                print("Prediction failed:", exc)
                return

            print(f"\n📸 {fname}")
            print(f"🔎 Used feature set: {used}")
            print(f"✅ Prediction: {'SHIP' if label == 1 else 'NO SHIP'}")
            print(f"🧮 Confidence (ship probability): {prob:.4f}")
            display(pil)

    uploader.observe(on_upload_change, names='value')

# Run the widget
upload_and_predict_widget(img_size=256)
